# Stickler as a Strands Evals evaluator

Strands Evals scores structured output with `Equals`: whole-object `==`, so 0.0 or 1.0. Stickler scores
the same output field by field, so the number says *how* wrong it is and *which* field to fix.

Offline and deterministic. No credentials, no model calls. For the same evaluator against a real
dataset with live Bedrock extraction, see `Strands_Evals_FCC_Demo.ipynb`.

## Setup

In [1]:
import datetime
from typing import List, Optional

from pydantic import BaseModel, Field
from strands_evals import Case, Experiment
from strands_evals.evaluators import Equals

from stickler.integrations.strands_evals import StructuredOutputEvaluator

## The model and the data

A plain Pydantic model, the kind an agent already uses for `structured_output_model`. No stickler
annotations. Nullable fields are `Optional` because on a real invoice they are legitimately absent.

In [2]:
class LineItem(BaseModel):
    sku: Optional[str] = None
    description: Optional[str] = None
    unit_price: Optional[float] = None


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: Optional[datetime.date] = None
    total_amount: Optional[float] = None
    line_items: List[LineItem] = Field(default_factory=list)


def inv(iid, vendor, date, total, items):
    return Invoice(invoice_id=iid, vendor_name=vendor, invoice_date=date,
                   total_amount=total, line_items=[LineItem(**i) for i in items])


GROUND_TRUTH = {
    "perfect":    inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
                      [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}]),
    "case-only":  inv("INV-002", "Beta Industries", "2026-02-01", 220.00,
                      [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}]),
    "amount-off": inv("INV-003", "Gamma Ltd", "2026-02-20", 310.00,
                      [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}]),
    "missing":    inv("INV-004", "Delta LLC", "2026-03-05", 90.00,
                      [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}]),
    "extra-line": inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
                      [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00}]),
    "wrong":      inv("INV-006", "Zeta Holdings", "2026-04-02", 75.00,
                      [{"sku": "SKU-6", "description": "Bracket", "unit_price": 75.00}]),
}

PREDICTIONS = {
    # exact match
    "perfect":    inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
                      [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}]),
    # vendor differs only in case
    "case-only":  inv("INV-002", "BETA INDUSTRIES", "2026-02-01", 220.00,
                      [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}]),
    # total slightly off
    "amount-off": inv("INV-003", "Gamma Ltd", "2026-02-20", 311.50,
                      [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}]),
    # date not extracted at all
    "missing":    inv("INV-004", "Delta LLC", None, 90.00,
                      [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}]),
    # hallucinated a second line item
    "extra-line": inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
                      [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00},
                       {"sku": "SKU-9", "description": "Phantom", "unit_price": 12.00}]),
    # wrong on nearly everything
    "wrong":      inv("INV-999", "Omega Group", "2025-11-11", 12.00,
                      [{"sku": "SKU-X", "description": "Unrelated", "unit_price": 12.00}]),
}

cases = [Case(name=k, input="", expected_output=GROUND_TRUTH[k], metadata={"k": k})
         for k in GROUND_TRUTH]
def task(case):
    return PREDICTIONS[case.metadata["k"]]

print(f"{len(cases)} cases")

6 cases


## Equals versus stickler

Same cases, same harness, same predictions. Only the evaluator differs.

In [3]:
evaluator = StructuredOutputEvaluator(Invoice)

stickler_report = await Experiment(cases=cases, evaluators=[evaluator]).run_evaluations_async(task)
equals_report = await Experiment(cases=cases, evaluators=[Equals()]).run_evaluations_async(task)

print(f"{'case':12} {'stickler':>9} {'equals':>7}")
print("-" * 30)
for c, s, e in zip(stickler_report.cases, stickler_report.scores, equals_report.scores):
    print(f"{c['name']:12} {s:>9.3f} {e:>7.1f}")

print(f"\noverall      {stickler_report.overall_score:>9.3f} {equals_report.overall_score:>7.3f}")
print(f"distinct     {len(set(round(v, 4) for v in stickler_report.scores)):>9} "
      f"{len(set(round(v, 4) for v in equals_report.scores)):>7}")

case          stickler  equals
------------------------------
perfect          1.000     1.0
case-only        1.000     0.0
amount-off       0.800     0.0
missing          0.800     0.0
extra-line       0.900     0.0
wrong            0.025     0.0

overall          0.754   0.167
distinct             4       2


`Equals` collapses five of six documents to 0.0, so it cannot rank extractors or spot a regression.
Stickler separates "one field slightly off" from "wrong on everything".

## Per-case detail

`evaluate()` returns one `EvaluationOutput` per top-level field, so field detail is already in the
report. The row labelled `__overall__` carries the weighted case score.

In [4]:
for entry in evaluator.per_case():
    print(f"{entry['case']:12} {entry['overall_score']:>5.2f}  "
          f"pass={str(entry['matched']):5}  {entry['field_scores']}")

wrong         0.03  pass=False  {'invoice_id': 0.0, 'vendor_name': 0.0, 'invoice_date': 0.0, 'total_amount': 0.0, 'line_items': 0.125}
perfect       1.00  pass=True   {'invoice_id': 1.0, 'vendor_name': 1.0, 'invoice_date': 1.0, 'total_amount': 1.0, 'line_items': 1.0}
missing       0.80  pass=False  {'invoice_id': 1.0, 'vendor_name': 1.0, 'invoice_date': 0.0, 'total_amount': 1.0, 'line_items': 1.0}
amount-off    0.80  pass=False  {'invoice_id': 1.0, 'vendor_name': 1.0, 'invoice_date': 1.0, 'total_amount': 0.0, 'line_items': 1.0}
extra-line    0.90  pass=True   {'invoice_id': 1.0, 'vendor_name': 1.0, 'invoice_date': 1.0, 'total_amount': 1.0, 'line_items': 0.5}
case-only     1.00  pass=True   {'invoice_id': 1.0, 'vendor_name': 1.0, 'invoice_date': 1.0, 'total_amount': 1.0, 'line_items': 1.0}


## Dataset rollup

`metrics()` returns stickler's five-category confusion matrix per field path, including nested paths.
It runs no extra comparisons: each case was compared once above and the raw result was kept.

The five categories separate failure modes that a single score cannot. **FN** is a field the extractor
missed, **FA** is one it invented, **FD** is one it got wrong.

In [5]:
rollup = evaluator.metrics()["Invoice"]

print(f"documents: {rollup.document_count}\n")
print(f"{'field':26} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 62)
for path, m in sorted(rollup.field_metrics.items(),
                      key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
    print(f"{path:26} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} {m.get('fd', 0):>3}"
          f"  {m.get('cm_precision', 0):>5.2f} {m.get('cm_recall', 0):>5.2f} {m.get('cm_f1', 0):>5.2f}")

documents: 6

field                       tp  fn  fa  fd   prec   rec    f1
--------------------------------------------------------------
total_amount                 4   0   0   2   0.67  1.00  0.80
invoice_date                 4   1   0   1   0.80  0.80  0.80
line_items                   5   0   1   1   0.71  1.00  0.83
invoice_id                   5   0   0   1   0.83  1.00  0.91
line_items.description       5   0   1   0   0.83  1.00  0.91
line_items.sku               5   0   1   0   0.83  1.00  0.91
line_items.unit_price        5   0   1   0   0.83  1.00  0.91
vendor_name                  5   0   0   1   0.83  1.00  0.91


Read the nested rows carefully. A `line_items.*` row only counts documents whose line-item pair scored
at or above `match_threshold`. Below that, threshold gating treats the pair as atomic and emits no
field breakdown, so those documents appear as `fd` on `line_items` and are absent from the child rows.
Child rows therefore have a smaller denominator than the document count.

Nested leaves carry counts and precision/recall/F1 but no mean score
([#249](https://github.com/awslabs/stickler/issues/249)). Row sets are data-dependent, so use `.get()`
rather than indexing.

## Why each field scored that way

Nothing was configured, so every comparator and threshold was inferred from the model. `explain()`
shows what was chosen and on what basis, which is what makes a score defensible.

In [6]:
print(f"{'field':26} {'comparator':24} {'thr':>5}  basis")
print("-" * 68)
for path, cfg in evaluator.explain().items():
    print(f"{path:26} {cfg['comparator']:24} {cfg['threshold']:>5}  {cfg['source']}")

field                      comparator                 thr  basis
--------------------------------------------------------------------
invoice_id                 ExactComparator            1.0  name-token
vendor_name                LevenshteinComparator     0.85  name-token
invoice_date               DateComparator            0.95  name-token
total_amount               NumericComparator         0.95  name-token
line_items                 Hungarian (per-element StructuredModel)   0.7  type
line_items.sku             ExactComparator            1.0  name-token
line_items.description     FuzzyComparator            0.6  name-token
line_items.unit_price      NumericComparator         0.95  name-token


## Mixed output types

`model_cls` is optional. Omit it and the model class is inferred per case, and `metrics()` partitions
its rollup by class. This matters because feeding two schemas into one rollup would union their field
paths, making a field present in half the documents look missed in the rest.

In [7]:
class Receipt(BaseModel):
    merchant: str
    tax: float


mixed_gt = {"inv-1": GROUND_TRUTH["perfect"], "rec-1": Receipt(merchant="Corner Store", tax=4.50)}
mixed_pred = {"inv-1": PREDICTIONS["perfect"], "rec-1": Receipt(merchant="Corner Store", tax=9.99)}
mixed_cases = [Case(name=k, input="", expected_output=mixed_gt[k], metadata={"k": k}) for k in mixed_gt]

mixed_eval = StructuredOutputEvaluator()          # no model_cls
await Experiment(cases=mixed_cases, evaluators=[mixed_eval]).run_evaluations_async(
    lambda c: mixed_pred[c.metadata["k"]])

for model_name, pe in mixed_eval.metrics().items():
    print(f"{model_name:10} docs={pe.document_count}  fields={sorted(pe.field_metrics)}")

Invoice    docs=1  fields=['invoice_date', 'invoice_id', 'line_items', 'line_items.description', 'line_items.sku', 'line_items.unit_price', 'total_amount', 'vendor_name']
Receipt    docs=1  fields=['merchant', 'tax']


Pass `model_cls` instead and the evaluator is strict: anything that will not validate as that class
raises, which is what you want for a single-schema suite.

## Notes for reuse

The evaluator accumulates across cases, so call `reset()` before reusing an instance for a second run.

It is safe at any concurrency. During a run it only appends to a list, which is atomic under the GIL,
so it holds no locks. All aggregation happens in `metrics()`, after the run.

In [8]:
print(f"before reset: {rollup.document_count} documents")
evaluator.reset()
print(f"after reset:  {evaluator.metrics()}")

before reset: 6 documents
after reset:  {}
